# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashelesc/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
- Ranking Signals Analysis(my lane).
- This is a ranking task, because the only output that matters is which pages sit at the top of the list. A team can only act as on a fixed number of pages per week, so relative order matters more than the exact score attached to each page.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

## 2. Target or proxy

- **is_declining_label** is the binary summary of **trend_direction**, it uses **trend_direction** to decide if each row's label should be **0** or **1**.
- **trend_direction** in turn thresholds **trend_pct**.
- **trend_direction** has 5 categories {**down**, **up**, **stable**, **flat**, **new**}.
- **new** and **flat** both have only 0's in the **trend_pct** feature, so I have dropped both from the entire dataset.
- **stable** has undergone slight changes, so I have flagged it as **not declining**, alongside **up**.
- **1** indicates that the page is flagged as **declining**, (**trend_pct** = **down**).
- **0** indicates that the page is flagged as **not declining**, (**trend_pct** = **up**).

In [3]:
print(df.trend_direction.value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [4]:
# descriptive statistics of trend_pct = 'new'
print(df[df['trend_direction'] == 'new']['trend_pct'].describe())

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: trend_pct, dtype: float64


In [5]:
# descriptive statistics of trend_pct = 'stable'
print(df[df['trend_direction'] == 'stable']['trend_pct'].describe())

count    5962.000000
mean       -3.185944
std        11.054723
min       -20.000000
25%       -12.700000
50%        -3.800000
75%         5.000000
max        20.000000
Name: trend_pct, dtype: float64


In [6]:
# descriptive statistics of trend_pct = 'flat'
print(df[df['trend_direction'] == 'flat']['trend_pct'].describe())

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: trend_pct, dtype: float64


In [7]:
# rows to remove {trend_pct: 'new', 'flat'}
labeled = df[~df['trend_direction'].isin(['new', 'flat'])].copy()
labeled['is_declining_label'] = (labeled['trend_direction'] == 'down').astype(int)

print(f"Excluded {len(df) - len(labeled)} rows")
print()
print(labeled['is_declining_label'].value_counts(normalize = True))

Excluded 3388 rows

is_declining_label
1    0.611078
0    0.388922
Name: proportion, dtype: float64


## 3. Success metric
- Precision@K of the top K pages a ranking flags, what fraction are genuinely declining?

In [8]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]

    return topk.mean()

print("Hand rule: Precision@20 -> 0.900, Precision@50 -> 0.680")
print("Best Decision Tree: Precision@20 -> 0.850, Precision@50 -> 0.740")

Hand rule: Precision@20 -> 0.900, Precision@50 -> 0.680
Best Decision Tree: Precision@20 -> 0.850, Precision@50 -> 0.740


## 4. The unit of analysis, as a real dataframe

- One row represents one content page (**content_id**), pages belong to one of 32 clients (**client_id**).
- This means that any **train/test** split must be done by **client_id** to avoid data leakage between splits.

In [9]:
print(df.shape[0])
print(df.content_id.nunique())
print(df.client_id.nunique())

30000
30000
32


## 5. Why ML beats a fixed rule here
- A fixed rule beat a shallow tree at **Precision@20** in notebook 2 (0.900 vs 0.550 to 0.850, depending on the tree's depth and features.
- On the corrected labeled set, after having excluded both **new** and **flat**, which had no **trend_pct** to threshold.
- The best tree scores **Precision@20**: 0.750, **Precision@50**: 0.800.
- Where ML earns it's place isn't raw accuracy over a hand rule, it's that it discovers which signals matter (**avg_position** and **days_since_last_update** over **impressions_90d**) without manual threshold-picking, and it does not collapse when the label definition gets refined, a fixed rule would need to be manually re-tuned every time the underlying data cleaning changes, as it did here.

In [10]:
from sklearn.tree import DecisionTreeClassifier

features = ['content_age_days', 'days_since_last_update', 'avg_position', 'ctr', 'word_count']
X = labeled[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = labeled['is_declining_label']

tree = DecisionTreeClassifier(max_depth = 3, class_weight = 'balanced', random_state = 42)
tree.fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]

for k in (20, 50):
    print(f"K = {k} tree Precision@{k}: {precision_at_k(tree_score, y, k):.3f}")

K = 20 tree Precision@20: 0.750
K = 50 tree Precision@50: 0.800


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.